# House Prices - Advanced Regression Techniques
해당 제목은 아마 발전된 회귀를 이용해서 집값을 예측하는 프로젝트로 보입니다.

[캐글 링크](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview)



In [1]:
from pathlib import Path

from scipy._external.pyprima.common import ratio

# 해당 위치에 "사용자명/.kaggle/kaggle.json" 에 Legacy Api Key가 있는지 확인해야합니다. 아니면 사용할 수 없더라고요.

print(Path.home())
print((Path.home() / ".kaggle" / "kaggle.json").exists())

C:\Users\hurwa
True


In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: C:\Users\hurwa\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques


In [3]:
import pprint

# pathlib를 이용해서 해당 디렉토리에 데이터가 있는지 확인해줍니다.
data_dir = Path(path)

pprint.pprint(list(data_dir.iterdir()))

[WindowsPath('C:/Users/hurwa/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques/data_description.txt'),
 WindowsPath('C:/Users/hurwa/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques/sample_submission.csv'),
 WindowsPath('C:/Users/hurwa/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques/test.csv'),
 WindowsPath('C:/Users/hurwa/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques/train.csv')]


디렉토리 경로에 `iterdir()`을 통해 제너레이터를 `list()`로 뽑아보면 `data_description.txt`, `sample_submission.csv`, `test.csv`,`train.csv`를 확인할 수 있습니다.

In [4]:
import pandas as pd

train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")

display(test_df.head())
print(test_df.shape)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


(1459, 80)


In [5]:
print(train_df.shape)

# print(test_df.columns)
# print(train_df.columns)

for tr_c in list(train_df.columns):
    exists_flag=False
    for t_c in list(test_df.columns):
        if(str(t_c) == str(tr_c)):
            exists_flag=True
    if not exists_flag:
        print(c)

(1460, 81)


NameError: name 'c' is not defined

해당 데이터셋에서 Kaggle competition에서 제공하는 코드를 이용해서 데이터를 받게된다면 `.cache/kagglehub/competitions`에 데이터셋 폴더가 생성되는 것을 확인할 수 있습니다.

해당 집값 예측의 경우에는 `train.csv`, `test.csv`가 존재하며 `train`에는 `target` 값이 존재하는 것을 확인할 수 있습니다. `SalePrice`는 `train.csv` 데이터셋에만 존재하는 것을 확인할 수 있습니다.

> 다음에는 `data_description.txt`를 한번 확인해보도록 하겠습니다.

In [ ]:
with open(Path(data_dir / "data_description.txt"), "r") as f:
    print(f.read())

솔직히 뭐라고 하는지 잘 모르겠습니다...

GPT에게 도움을 받아보겠습니다.

---

일단 물어보니 데이터 사전인건 맞는거같은데 답변은 나왔지만 일단 할 수 있는 만큼만 한번 읽어보겠습니다.

좀 txt를 들여다보니 depth별로 바깥쪽은 컬럼명, 안쪽은 그 컬럼의 범주 정도를 나타내는 것 같습니다.

양이 적지는 않네요. 한번 요약해보면서 덩강덩강 읽어보겠습니다.

- MSSubClass는 아마 세일하는 주거지로 보입니다. 그런데 데이터만 보면 아마 STORY? AGES? 년도? 같은 데이터가 존재하네요. (MSZongoing를 보니까 sale가 세일이 아니라 판매를 의미하는 거였네요. 그리고 아마 얼마나 오래되었는지를 나타내느게 아닐까 싶습니다.)
- MSZongoing에서는 지역별로 나눠서 보여주는 것 같습니다. A는 예를 들어서 문화, C는 사업, FV는 뭔가 산업 관련같은데 이것도 조금 도움이 필요할 것 같습니다.
- LotFrontage의 경우에는 얼마나 거리가 긴지? 나타내는 모양인데 아마 범주형은 아니고 숫자형일 것으로 예측됩니다.
- LotArea: 뭔가 길이와 관련된거같은데 square라고 나와있기 때문에 면적을 말하는거같습니다. 구글 번역기에서는 피트라고 하는데 그것도 잘 모르겠습니다. Lot이 정확히 무엇인지
- Street는 그냥 자갈/포장을 의미하는 Grvl, Pave를 의미하네요.
- Alley의 경우에는 본적이 있는데 기억이 안나서 찾아보니 **골목**이네요.
- LotShape의 경우에도 Irregular이라는 규칙성을 나타내는 것으로 보아 약간 정사각형 모음과 같이 정렬되어있는지 아니면 구불구불 짜여져있는지를 확인하는 것 같습니다.

---

# 문제 발생
지금 영어가 안됩니다. 이거 GPT한테 던져주고 해석해달라 한다음에 모르는 문법 정리 한번 해야할 것 같습니다. 일단 읽을줄은 아는데 단어를 너무 모릅니다. 공부하고 진행하겠습니다.

In [6]:
# 일부 공부하고 작업 식작
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [7]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

# EDA
일단 총 데이터는 0 ~ 1459번까지 1460개 존재합니다.

식별자, 땅의 타입, 면적, 건축의 형태, 화덕? 지하실, 울타리 여부 등이 존재합니다.

일단 결측치 상태부터 한번 확인해보겠습니다.

In [10]:
train_df.isna().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

결측치가 있는 데이터들을 살펴보면

In [29]:
isna_series = (
    train_df.isna().sum()[train_df.isna().sum() > 0]
    .sort_values(ascending=False)
)

In [46]:
isna_columns = list(
    train_df.isna().sum()[train_df.isna().sum() > 0]
    .sort_values(ascending=False)
    .index
)

In [50]:
isna_columns

['PoolQC',
 'MiscFeature',
 'Alley',
 'Fence',
 'MasVnrType',
 'FireplaceQu',
 'LotFrontage',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'BsmtExposure',
 'BsmtFinType2',
 'BsmtQual',
 'BsmtCond',
 'BsmtFinType1',
 'MasVnrArea',
 'Electrical']

In [32]:
isna_count = train_df.isna().sum()

isna_df = pd.DataFrame({
    "count": isna_count[isna_count > 0],
    "raito": train_df.isna().mean()[isna_count > 0],
    "dtype": train_df.dtypes[isna_count > 0],
    "nunique": train_df.nunique()[isna_count > 0]
}).sort_values("raito", ascending=False)

In [33]:
isna_df

,count,raito,dtype,nunique
PoolQC,1453,0.995205,str,3
MiscFeature,1406,0.963014,str,4
Alley,1369,0.937671,str,2
Fence,1179,0.807534,str,4
MasVnrType,872,0.597260,str,3
FireplaceQu,690,0.472603,str,5
LotFrontage,259,0.177397,float64,110
GarageType,81,0.055479,str,6
GarageYrBlt,81,0.055479,float64,97
GarageFinish,81,0.055479,str,3


# 결측치 데이터 분석
데이터에 결측은 2가지로 나뉠 수 있습니다.
1. 실제 결측: 원래 값이 있어야하는데 기록되지 않음
2. 구조적인 없음: 그냥 없음을 NA로 표현한 것

In [49]:
for c in isna_columns:
    print(f"# [{c}]")
    print(train_df[c].isna().sum())
    if not len(train_df[c].unique()) > 20:
        print(train_df[c].unique())
    else:
        print(train_df[c].unique()[:21])
    print(train_df[c].dtype)
    print()

# [PoolQC]
1453
<StringArray>
[nan, 'Ex', 'Fa', 'Gd']
Length: 4, dtype: str
str

# [MiscFeature]
1406
<StringArray>
[nan, 'Shed', 'Gar2', 'Othr', 'TenC']
Length: 5, dtype: str
str

# [Alley]
1369
<StringArray>
[nan, 'Grvl', 'Pave']
Length: 3, dtype: str
str

# [Fence]
1179
<StringArray>
[nan, 'MnPrv', 'GdWo', 'GdPrv', 'MnWw']
Length: 5, dtype: str
str

# [MasVnrType]
872
<StringArray>
['BrkFace', nan, 'Stone', 'BrkCmn']
Length: 4, dtype: str
str

# [FireplaceQu]
690
<StringArray>
[nan, 'TA', 'Gd', 'Fa', 'Ex', 'Po']
Length: 6, dtype: str
str

# [LotFrontage]
259
[ 65.  80.  68.  60.  84.  85.  75.  nan  51.  50.  70.  91.  72.  66.
 101.  57.  44. 110.  98.  47. 108.]
float64

# [GarageType]
81
<StringArray>
['Attchd', 'Detchd', 'BuiltIn', 'CarPort', nan, 'Basment', '2Types']
Length: 7, dtype: str
str

# [GarageYrBlt]
81
[2003. 1976. 2001. 1998. 2000. 1993. 2004. 1973. 1931. 1939. 1965. 2005.
 1962. 2006. 1960. 1991. 1970. 1967. 1958. 1930. 2002.]
float64

# [GarageFinish]
81
<StringArr

# PoolQC
- NA는 그냥 수영장 없음
- TA값이 없긴 한데 그냥 평범한 수영장은 없어서 그런건가
# MiscFeature
- 이건 기타 요소가 없다는 의미
# Alley
- 골목 포장이 없거나 골목이 없다는 뜻?
# Fence
- Fence가 없다는 뜻
# MasVnrType
- 아마 마감 종류인거같은데 None는 없다는 의미.
- 나머지 컬럼들은 뭘 의미하는건지 잘 모르겠음
# FireplaceQu
- 화덕이라고 해야하나 집 안에 있는 벽난로를 의미함.
- 이니셜별로 품질을 말하는 것 같다.
# LotFrontage
- 숫자형인데 없으니까 그냥 결측치인듯.
- 숫자의 의미는 뭔가 요소와 이어진 거리의 길이?
# GarageType
- 이것도 그냥 없다는 의미. 아래의 Garage* 형태의 컬럼도 모두 없는 것이라고 보면 됨.
    - GarageYrBlt
    - GarageFinish
    - GarageQual
    - GarageCond
# BsmtExposure
- 이것도 그냥 지하실이 없다는 의미 아래도 동일함
    - BsmtFinType2
    - BsmtQual
    - BsmtCond
    - BsmtFinType1
# MasVnrArea
- 결측치.
- 벽쪽 마감 범위를 말하는거 같음
# Electrical
- nan가 전기가 없다는 것인지 아니면 측정하지 못한건지 확실하지 않음.

# 데이터 전처리
이제 결측치의 종류를 먼저 나누었습니다.

여기에서 나온 결측치는 크게 2종류로
- 구조적 결측치 (데이터를 못 모은게 아니라 애초에 값이 존재하지 않아서 결측치인 값. ` Not Applicable`)
- `Actual Missing Value` - 실제 결측치 (진짜로 데이터를 모으지 못한 값)

가 있습니다. 제가 찾은 값들 중에서는 마감 면적(`MasVnrArea`), `Electrical` 정도가 수치적으로 채워야할 값인 듯 합니다.

이렇게 되면 이제 다른 값들은 올바른 값으로 문자열 또는 수치형 범주 (MSSubClass)를 문자로 바꿔준 뒤 실제 결측 데이터의 경우에는 값을 적절히 찾아주고 (중앙값 보다는 transform을 이용해서), 범주형의 경우에는 타입에 따라 각각 비교가 불가능한 OneHot인코딩 또는 좋고 나쁨 또는 크고 작음 과 같은 성질이 있어 이를 비교가 가능한 형태인 Ordinal 인코딩을 할 수도 있습니다.

> 이때 MasVnrArea는 MasVnrType에 따라 마감을 하지 않은 것일 수 있습니다. 이 차이를 유심히 확인해야합니다.

이제 변환기를 천천히 만들어보겠습니다.

# 2. 전처리 시작

In [56]:
train_df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [55]:
# 값을 나누어주기
X = train_df.drop(columns=["SalePrice"])
y = train_df["SalePrice"]

X.shape, y.shape

((1460, 80), (1460,))

In [60]:
# 다음 셀에서 나타나게 될 transform을 위한 컬럼의 고유한 데이터 종류 확인하기
# 너무 많거나 적으면 다른걸 고려해야하는데 25면 대략 중앙값보다 나은 값을 도출할 수 있을 것 같음
X["Neighborhood"].unique()

<StringArray>
['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst',  'NWAmes',
 'OldTown', 'BrkSide',  'Sawyer', 'NridgHt',   'NAmes', 'SawyerW',  'IDOTRR',
 'MeadowV', 'Edwards',  'Timber', 'Gilbert', 'StoneBr', 'ClearCr', 'NPkVill',
 'Blmngtn',  'BrDale',   'SWISU', 'Blueste']
Length: 25, dtype: str

# 계획 세우기
- 데이터 없는거 다시 한번 확인해보기

- MSSubClass를 범주형 문자열로 일단 만들어주기
- 이건 일단 범주형으로 만들어서 경우에 따라 ordinal까지 고려해볼 사항

- 숫자형 예측해야하게 될 값
- MasVnrArea와 Electrical 정도를 예측하게 될거같은데 일단 Neighboorhood를 이용하여 중앙값을 내주는게 적절할 것 같고
- MasVnrArea의 경우에는 MasVnrType를 기반으로 해당 값이 결측치인 경우에는 0으로 설정해주는 것이 좋을거같습니다.

- 원핫 인코딩만 하면 될 값

- ordinal 인코딩 해야할 값

- 추가로 YrSold가 아마 판매되고 있던 시각을 나타내는 것 같으니까 거기에서 Year* 컬럼들을 이용하면 데이터가 조금 더 효과적으로 피처 엔지니어링을 할 수 있을 것 같습니다.

---

## 코덱스의 보정

1. 흐름은 괜찮은거 같고 train을 이용해서 fit를 먼저 작업한 다음에 test에만 있는 결측치를 고려해서 작업을 해야한다고 합니다.
2. 지역별 중앙값은 LotGrgontage정도로 하며 너무 난발하면 SalePrice같은 target를 학습할 수 있기 때문에 조심해야한다고 합니다.
3. 데이터 형태를 한번 볼 필요가 있긴 합니다.
4. 베이스라인 모델을 먼저 만들라고 합니다.

> 베이스라인 모델이란 **아무 튜닝도 거의 하지 않고, 최소한의 전처리만 해서 만든 첫 번째 기준 모델**을 말한다고 합니다.
> 베이스라인 모델이 있어야 피처 엔지니어링을 통해 학습의 결과가 좋아진 것이 맞는지 알 수 있게 됩니다.


In [62]:
column_note = {
    "LotFrontage": "도로와 접한 길이",
    "LotArea": "대지 면적",
    "Neighborhood": "동네/지역",
    "OverallQual": "전체 재료/마감 품질",
    "OverallCond": "전체 상태",
    "YearBuilt": "건축 연도",
    "GrLivArea": "지상층 거주 면적",
    "GarageCars": "차고 수용 차량 수",
    "GarageArea": "차고 면적",
}

In [ ]:
# 베이스라인 모델 만들어두기
def check_missing(df):
    missing = df.isna().sum()
    ratio = df.isna().mean()

    result = pd.DataFrame({
        "missing_counut": missing,
        "missing_ratio": ratio
    })

    return result[result["missing_count"] > 0].sort_values(
        "missing_count",
        ascending=False
    )

In [ ]:
def basic_preprocess(df):
    """값들을 기본적으로 'None' 또는 중앙값 등으로 빠르게 채워주는 전처리 기계"""
    df = df.copy()

    df = df.drop(columns=["Id"], errors="ignoroe")

    # 숫자형 데이터들.
    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
    categorical_cols

    for c in df.columns:

